In [16]:
%%bash
export WANDB_API_KEY="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"
set -euo pipefail
PROJ="/engram/nklab/am6949/SimMIM_ver2"
LOGDIR="${PROJ}/logs"
mkdir -p "${LOGDIR}"

set -x

find_simmim_python () {
  local candidates=(
    "${HOME}/.conda/envs/simmim/bin/python"
    "/home/${USER}/.conda/envs/simmim/bin/python"
    "/home/${USER}/miniconda3/envs/simmim/bin/python"
    "/home/${USER}/anaconda3/envs/simmim/bin/python"
  )
  for p in "${candidates[@]}"; do
    if [ -x "$p" ]; then
      echo "$p"
      return 0
    fi
  done
  return 1
}

PY="$(find_simmim_python || true)"
echo "[INFO] python candidate: ${PY:-<none>}"
test -n "${PY}" || { echo "[FATAL] Could not find simmim python." >&2; exit 127; }

# ----------------------------
# Configs / scripts
# ----------------------------
PRE_CFG="${PROJ}/configs/vit_base__800ep/simmim_pretrain__vit_base__img224__800ep.yaml"
FT_CFG="${PROJ}/configs/vit_base__800ep/simmim_finetune__vit_base__img224__800ep.yaml"

PRE_MAIN="${PROJ}/main_simmim_blur.py"
FT_MAIN="${PROJ}/main_finetune_blur.py"

IMNET_ROOT="/share/data/imagenet-pytorch"

# outputs
SWEEP_OUT="${PROJ}/model_weights_ver4/SimMIM_sweeps"
FT_OUT="${PROJ}/model_weights_ver4/finetune"
mkdir -p "${SWEEP_OUT}" "${FT_OUT}"

# ----------------------------
# Sweep definition
# ----------------------------
SEED="0"
DEPTHS=(6 3)
RATIOS=(0.65 0.95)

BATCH_SIZE="64"

GAUSS_SCALE="2.0"
GAUSS_SIGMA_CI_32="1.0"

PRE_EPOCHS="100"
FT_EPOCHS="100"

# ----------------------------
# depth-specific PRETRAIN hyperparams
# ----------------------------
D6_PRE_WARMUP_EPOCHS="10"
D6_PRE_BASE_LR="2.0e-4"
D6_PRE_WARMUP_LR="1.0e-6"
D6_PRE_DROP_PATH="0.10"

D3_PRE_WARMUP_EPOCHS="${D6_PRE_WARMUP_EPOCHS}"
D3_PRE_BASE_LR="${D6_PRE_BASE_LR}"
D3_PRE_WARMUP_LR="${D6_PRE_WARMUP_LR}"
D3_PRE_DROP_PATH="${D6_PRE_DROP_PATH}"

# ----------------------------
# depth-specific FINETUNE hyperparams
# ----------------------------
D6_WARMUP_EPOCHS="20"
D6_BASE_LR="1.50e-3"
D6_WARMUP_LR="3.0e-6"
D6_MIN_LR="1.0e-6"
D6_WEIGHT_DECAY="0.05"
D6_LAYER_DECAY="0.65"
D6_DROP_PATH="0.10"

D3_WARMUP_EPOCHS="${D6_WARMUP_EPOCHS}"
D3_BASE_LR="${D6_BASE_LR}"
D3_WARMUP_LR="${D6_WARMUP_LR}"
D3_MIN_LR="${D6_MIN_LR}"
D3_WEIGHT_DECAY="${D6_WEIGHT_DECAY}"
D3_LAYER_DECAY="${D6_LAYER_DECAY}"
D3_DROP_PATH="${D6_DROP_PATH}"

RATIOS_STR="${RATIOS[*]}"

write_job () {
  local DEPTH="$1"
  local JOBTAG="d${DEPTH}_gauss_nocut_loop"
  local JOBFILE="${PROJ}/JOB_DDP_${JOBTAG}.sh"

  cat > "${JOBFILE}" <<'EOS'
#!/bin/bash
#SBATCH --job-name=__JOBTAG__
#SBATCH --partition=nklab
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=12
#SBATCH --nodelist=ax26
#SBATCH --mem=64G
#SBATCH --time=10-00:00:00
#SBATCH --output=__LOGDIR__/__JOBTAG___%j.log
#SBATCH --error=__LOGDIR__/__JOBTAG___%j.err

set -euo pipefail
unset MASTER_ADDR MASTER_PORT

echo "[INFO] JOBTAG=__JOBTAG__"
echo "[INFO] DEPTH=__DEPTH__"
echo "[INFO] SEED=__SEED__"
echo "[INFO] RATIOS=__RATIOS_STR__"
echo "[INFO] BATCH_SIZE=__BATCH_SIZE__"
echo "[INFO] CORR=gauss (FFT)  NO-CUTOFF (cutoff=None via config default; launcher does not override)"
echo "[INFO] sigma_ci(32)=__GAUSS_SIGMA_CI_32__  scale=__GAUSS_SCALE__"
echo "[INFO] Host=$(hostname)"
echo "[INFO] SLURM_JOB_ID=${SLURM_JOB_ID}"
echo "[INFO] Date=$(date)"
echo ""

find_simmim_python () {
  local candidates=(
    "${HOME}/.conda/envs/simmim/bin/python"
    "/home/${USER}/.conda/envs/simmim/bin/python"
    "/home/${USER}/miniconda3/envs/simmim/bin/python"
    "/home/${USER}/anaconda3/envs/simmim/bin/python"
  )
  for p in "${candidates[@]}"; do
    if [ -x "$p" ]; then
      echo "$p"
      return 0
    fi
  done
  return 1
}

PY="$(find_simmim_python || true)"
if [ -z "${PY}" ]; then
  echo "[FATAL] Could not find simmim python." >&2
  exit 127
fi
echo "[INFO] Using PY=${PY}"
"${PY}" -V || true
echo ""

nvidia-smi || true
echo ""

NGPU="${SLURM_GPUS_ON_NODE:-1}"
if [ -n "${SLURM_JOB_GPUS:-}" ]; then
  NGPU="$("${PY}" - <<'PY'
import os
s=os.environ.get("SLURM_JOB_GPUS","").strip()
if not s:
    print(1)
else:
    parts=[p for p in s.replace(" ",",").split(",") if p!=""]
    print(len(parts) if parts else 1)
PY
)"
fi
echo "[INFO] SLURM_GPUS_ON_NODE=${SLURM_GPUS_ON_NODE:-<unset>}  SLURM_JOB_GPUS=${SLURM_JOB_GPUS:-<unset>}  -> NGPU=${NGPU}"

export OMP_NUM_THREADS=1
export MKL_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1
export NUMEXPR_NUM_THREADS=1
export PYTHONPATH="__PROJ__:${PYTHONPATH:-}"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

export PROJ="__PROJ__"
export PRE_CFG="__PRE_CFG__"
export FT_CFG="__FT_CFG__"
export PRE_MAIN="__PRE_MAIN__"
export FT_MAIN="__FT_MAIN__"
export IMNET_ROOT="__IMNET_ROOT__"
export SWEEP_OUT="__SWEEP_OUT__"
export FT_OUT="__FT_OUT__"

export GAUSS_SCALE="__GAUSS_SCALE__"
export GAUSS_SIGMA_CI_32="__GAUSS_SIGMA_CI_32__"

export PRE_EPOCHS="__PRE_EPOCHS__"
export FT_EPOCHS="__FT_EPOCHS__"
export SEED="__SEED__"
export DEPTH="__DEPTH__"
export BATCH_SIZE="__BATCH_SIZE__"

export D6_PRE_WARMUP_EPOCHS="__D6_PRE_WARMUP_EPOCHS__"
export D6_PRE_BASE_LR="__D6_PRE_BASE_LR__"
export D6_PRE_WARMUP_LR="__D6_PRE_WARMUP_LR__"
export D6_PRE_DROP_PATH="__D6_PRE_DROP_PATH__"

export D3_PRE_WARMUP_EPOCHS="__D3_PRE_WARMUP_EPOCHS__"
export D3_PRE_BASE_LR="__D3_PRE_BASE_LR__"
export D3_PRE_WARMUP_LR="__D3_PRE_WARMUP_LR__"
export D3_PRE_DROP_PATH="__D3_PRE_DROP_PATH__"

export D6_WARMUP_EPOCHS="__D6_WARMUP_EPOCHS__"
export D6_BASE_LR="__D6_BASE_LR__"
export D6_WARMUP_LR="__D6_WARMUP_LR__"
export D6_MIN_LR="__D6_MIN_LR__"
export D6_WEIGHT_DECAY="__D6_WEIGHT_DECAY__"
export D6_LAYER_DECAY="__D6_LAYER_DECAY__"
export D6_DROP_PATH="__D6_DROP_PATH__"

export D3_WARMUP_EPOCHS="__D3_WARMUP_EPOCHS__"
export D3_BASE_LR="__D3_BASE_LR__"
export D3_WARMUP_LR="__D3_WARMUP_LR__"
export D3_MIN_LR="__D3_MIN_LR__"
export D3_WEIGHT_DECAY="__D3_WEIGHT_DECAY__"
export D3_LAYER_DECAY="__D3_LAYER_DECAY__"
export D3_DROP_PATH="__D3_DROP_PATH__"

mkdir -p "${SWEEP_OUT}" "${FT_OUT}"
cd "${PROJ}"

SIGMA_CI="$("${PY}" - <<'PY'
import os
s32 = float(os.environ["GAUSS_SIGMA_CI_32"])
scale = float(os.environ["GAUSS_SCALE"])
print(s32 * scale)
PY
)"
echo "[INFO] FFT-Gauss scaled: sigma_ci=${SIGMA_CI}  cutoff_ci=None (not overridden)"

MS_PRE="$("${PY}" - <<'PY'
import os
E=int(os.environ["PRE_EPOCHS"])
print(int(round(0.875*E)))
PY
)"
MS_FT="$("${PY}" - <<'PY'
import os
E=int(os.environ["FT_EPOCHS"])
print(int(round(0.875*E)))
PY
)"
echo "[INFO] LR multistep: PRE=${MS_PRE} (PRE_EPOCHS=${PRE_EPOCHS}) | FT=${MS_FT} (FT_EPOCHS=${FT_EPOCHS})"

case "${DEPTH}" in
  6)
    PRE_WARMUP_EPOCHS="${D6_PRE_WARMUP_EPOCHS}"
    PRE_BASE_LR="${D6_PRE_BASE_LR}"
    PRE_WARMUP_LR="${D6_PRE_WARMUP_LR}"
    PRE_DROP_PATH="${D6_PRE_DROP_PATH}"

    FT_WARMUP_EPOCHS="${D6_WARMUP_EPOCHS}"
    FT_BASE_LR="${D6_BASE_LR}"
    FT_WARMUP_LR="${D6_WARMUP_LR}"
    FT_MIN_LR="${D6_MIN_LR}"
    FT_WEIGHT_DECAY="${D6_WEIGHT_DECAY}"
    FT_LAYER_DECAY="${D6_LAYER_DECAY}"
    FT_DROP_PATH="${D6_DROP_PATH}"
    ;;
  3)
    PRE_WARMUP_EPOCHS="${D3_PRE_WARMUP_EPOCHS}"
    PRE_BASE_LR="${D3_PRE_BASE_LR}"
    PRE_WARMUP_LR="${D3_PRE_WARMUP_LR}"
    PRE_DROP_PATH="${D3_PRE_DROP_PATH}"

    FT_WARMUP_EPOCHS="${D3_WARMUP_EPOCHS}"
    FT_BASE_LR="${D3_BASE_LR}"
    FT_WARMUP_LR="${D3_WARMUP_LR}"
    FT_MIN_LR="${D3_MIN_LR}"
    FT_WEIGHT_DECAY="${D3_WEIGHT_DECAY}"
    FT_LAYER_DECAY="${D3_LAYER_DECAY}"
    FT_DROP_PATH="${D3_DROP_PATH}"
    ;;
  *)
    echo "[FATAL] Unsupported DEPTH=${DEPTH} (expected 3 or 6)" >&2
    exit 2
    ;;
esac

for RATIO in __RATIOS_STR__; do
  RTAG="$(echo "${RATIO}" | sed 's/0\.//; s/\.//g')"
  PRE_WANDB_NAME="blur_nocut_d${DEPTH}_r${RTAG}_pre"
  FT_WANDB_NAME="blur_nocut_d${DEPTH}_r${RTAG}_ft"

  echo "============================================================"
  echo "[RUN] DEPTH=${DEPTH}  MASK_RATIO=${RATIO}  CORR=gauss (FFT)  NO-CUTOFF  SEED=${SEED}"
  echo "[W&B] PRE=${PRE_WANDB_NAME} | FT=${FT_WANDB_NAME}"
  echo "[RESOLVED] PRE: warmup_epochs=${PRE_WARMUP_EPOCHS} base_lr=${PRE_BASE_LR} warmup_lr=${PRE_WARMUP_LR} drop_path=${PRE_DROP_PATH}"
  echo "[RESOLVED]  FT: warmup_epochs=${FT_WARMUP_EPOCHS} base_lr=${FT_BASE_LR} warmup_lr=${FT_WARMUP_LR} min_lr=${FT_MIN_LR} wd=${FT_WEIGHT_DECAY} layer_decay=${FT_LAYER_DECAY} drop_path=${FT_DROP_PATH}"
  echo "============================================================"

  RUNNAME="vitb_img224_seed${SEED}_d${DEPTH}_r${RATIO}_gauss_nocut_s${SIGMA_CI}"
  PRE_OUT_DIR="${SWEEP_OUT}/${RUNNAME}"
  FT_OUT_DIR="${FT_OUT}/${RUNNAME}__FT_ep${FT_EPOCHS}"

  mkdir -p "${PRE_OUT_DIR}" "${FT_OUT_DIR}"

  OPTS_PRE=(
    SEED "${SEED}"
    TRAIN.EPOCHS "${PRE_EPOCHS}"

    MODEL.TYPE vit
    MODEL.VIT.DEPTH "${DEPTH}"
    MODEL.DROP_PATH_RATE "${PRE_DROP_PATH}"

    TRAIN.WARMUP_EPOCHS "${PRE_WARMUP_EPOCHS}"
    TRAIN.BASE_LR "${PRE_BASE_LR}"
    TRAIN.WARMUP_LR "${PRE_WARMUP_LR}"

    TRAIN.LR_SCHEDULER.NAME multistep
    TRAIN.LR_SCHEDULER.GAMMA 0.1
    TRAIN.LR_SCHEDULER.MULTISTEPS "[${MS_PRE}]"

    DATA.BLUR_TYPE gaussian
    DATA.BLUR_GAUSS_SIGMA_CI "${SIGMA_CI}"
  )

  echo "[INFO] PRE --opts: ${OPTS_PRE[*]}"

  "${PY}" -m torch.distributed.run --standalone --nproc_per_node="${NGPU}" "${PRE_MAIN}" \
    --cfg "${PRE_CFG}" \
    --data-path "${IMNET_ROOT}" \
    --output "${PRE_OUT_DIR}" \
    --tag sweep \
    --batch-size "${BATCH_SIZE}" \
    --wandb-run-name "${PRE_WANDB_NAME}" \
    --amp-opt-level O1 \
    --corruption blur \
    --mask-ratio "${RATIO}" \
    --loss-on-full-image \
    --opts ${OPTS_PRE[@]}

  CKPT=""
  if [ -f "${PRE_OUT_DIR}/simmim_pretrain/sweep/best.pt" ]; then
    CKPT="${PRE_OUT_DIR}/simmim_pretrain/sweep/best.pt"
  elif [ -f "${PRE_OUT_DIR}/simmim_pretrain/sweep/best.pth" ]; then
    CKPT="${PRE_OUT_DIR}/simmim_pretrain/sweep/best.pth"
  else
    CKPT="$(ls -1 "${PRE_OUT_DIR}"/simmim_pretrain/sweep/ckpt_epoch_* 2>/dev/null | tail -n 1 || true)"
  fi
  test -f "${CKPT}" || { echo "[FATAL] Missing checkpoint after pretrain: ${CKPT}" >&2; exit 3; }
  echo "[INFO] Using CKPT=${CKPT}"

  OPTS_FT=(
    SEED "${SEED}"
    TRAIN.EPOCHS "${FT_EPOCHS}"

    MODEL.TYPE vit
    MODEL.VIT.DEPTH "${DEPTH}"

    TRAIN.WARMUP_EPOCHS "${FT_WARMUP_EPOCHS}"
    TRAIN.BASE_LR "${FT_BASE_LR}"
    TRAIN.WARMUP_LR "${FT_WARMUP_LR}"
    TRAIN.MIN_LR "${FT_MIN_LR}"
    TRAIN.WEIGHT_DECAY "${FT_WEIGHT_DECAY}"
    TRAIN.LAYER_DECAY "${FT_LAYER_DECAY}"
    MODEL.DROP_PATH_RATE "${FT_DROP_PATH}"
  )

  echo "[INFO] FT --opts: ${OPTS_FT[*]}"

  "${PY}" -m torch.distributed.run --standalone --nproc_per_node="${NGPU}" "${FT_MAIN}" \
    --cfg "${FT_CFG}" \
    --data-path "${IMNET_ROOT}" \
    --pretrained "${CKPT}" \
    --output "${FT_OUT_DIR}" \
    --tag finetune \
    --batch-size "${BATCH_SIZE}" \
    --wandb-run-name "${FT_WANDB_NAME}" \
    --amp-opt-level O1 \
    --opts ${OPTS_FT[@]}

  echo "[DONE] DEPTH=${DEPTH} RATIO=${RATIO} finished."
done

echo "[DONE] __JOBTAG__ finished all ratios."
EOS

  sed -i \
    -e "s|__PROJ__|${PROJ}|g" \
    -e "s|__LOGDIR__|${LOGDIR}|g" \
    -e "s|__PRE_CFG__|${PRE_CFG}|g" \
    -e "s|__FT_CFG__|${FT_CFG}|g" \
    -e "s|__PRE_MAIN__|${PRE_MAIN}|g" \
    -e "s|__FT_MAIN__|${FT_MAIN}|g" \
    -e "s|__IMNET_ROOT__|${IMNET_ROOT}|g" \
    -e "s|__SWEEP_OUT__|${SWEEP_OUT}|g" \
    -e "s|__FT_OUT__|${FT_OUT}|g" \
    -e "s|__GAUSS_SCALE__|${GAUSS_SCALE}|g" \
    -e "s|__GAUSS_SIGMA_CI_32__|${GAUSS_SIGMA_CI_32}|g" \
    -e "s|__PRE_EPOCHS__|${PRE_EPOCHS}|g" \
    -e "s|__FT_EPOCHS__|${FT_EPOCHS}|g" \
    -e "s|__SEED__|${SEED}|g" \
    -e "s|__BATCH_SIZE__|${BATCH_SIZE}|g" \
    -e "s|__RATIOS_STR__|${RATIOS_STR}|g" \
    -e "s|__D6_PRE_WARMUP_EPOCHS__|${D6_PRE_WARMUP_EPOCHS}|g" \
    -e "s|__D6_PRE_BASE_LR__|${D6_PRE_BASE_LR}|g" \
    -e "s|__D6_PRE_WARMUP_LR__|${D6_PRE_WARMUP_LR}|g" \
    -e "s|__D6_PRE_DROP_PATH__|${D6_PRE_DROP_PATH}|g" \
    -e "s|__D3_PRE_WARMUP_EPOCHS__|${D3_PRE_WARMUP_EPOCHS}|g" \
    -e "s|__D3_PRE_BASE_LR__|${D3_PRE_BASE_LR}|g" \
    -e "s|__D3_PRE_WARMUP_LR__|${D3_PRE_WARMUP_LR}|g" \
    -e "s|__D3_PRE_DROP_PATH__|${D3_PRE_DROP_PATH}|g" \
    -e "s|__D6_WARMUP_EPOCHS__|${D6_WARMUP_EPOCHS}|g" \
    -e "s|__D6_BASE_LR__|${D6_BASE_LR}|g" \
    -e "s|__D6_WARMUP_LR__|${D6_WARMUP_LR}|g" \
    -e "s|__D6_MIN_LR__|${D6_MIN_LR}|g" \
    -e "s|__D6_WEIGHT_DECAY__|${D6_WEIGHT_DECAY}|g" \
    -e "s|__D6_LAYER_DECAY__|${D6_LAYER_DECAY}|g" \
    -e "s|__D6_DROP_PATH__|${D6_DROP_PATH}|g" \
    -e "s|__D3_WARMUP_EPOCHS__|${D3_WARMUP_EPOCHS}|g" \
    -e "s|__D3_BASE_LR__|${D3_BASE_LR}|g" \
    -e "s|__D3_WARMUP_LR__|${D3_WARMUP_LR}|g" \
    -e "s|__D3_MIN_LR__|${D3_MIN_LR}|g" \
    -e "s|__D3_WEIGHT_DECAY__|${D3_WEIGHT_DECAY}|g" \
    -e "s|__D3_LAYER_DECAY__|${D3_LAYER_DECAY}|g" \
    -e "s|__D3_DROP_PATH__|${D3_DROP_PATH}|g" \
    -e "s|__DEPTH__|${DEPTH}|g" \
    -e "s|__JOBTAG__|${JOBTAG}|g" \
    "${JOBFILE}"

  chmod +x "${JOBFILE}"
  echo "${JOBFILE}"
}

# ----------------------------
# generate jobs + submit
# ----------------------------
JOBFILES=()
for D in "${DEPTHS[@]}"; do
  JOBFILES+=( "$(write_job "${D}")" )
done

echo "[INFO] JOBFILES:"
printf '  %s\n' "${JOBFILES[@]}"

IDS=()
for jf in "${JOBFILES[@]}"; do
  test -f "${jf}" || { echo "[FATAL] Missing jobfile: ${jf}" >&2; exit 4; }
  IDS+=( "$(sbatch -p nklab --parsable "${jf}")" )
done

set +x
echo "Submitted 2 jobs on ax17 (seed=${SEED}):"
i=0
for D in "${DEPTHS[@]}"; do
  JOBTAG="d${D}_gauss_nocut_loop"
  ID="${IDS[$i]}"
  echo "  ${JOBTAG} : ${ID}"
  echo "    tail -n 200 -F ${LOGDIR}/${JOBTAG}_${ID}.log"
  echo "    tail -n 200 -F ${LOGDIR}/${JOBTAG}_${ID}.err"
  i=$((i+1))
done

++ find_simmim_python
++ candidates=("${HOME}/.conda/envs/simmim/bin/python" "/home/${USER}/.conda/envs/simmim/bin/python" "/home/${USER}/miniconda3/envs/simmim/bin/python" "/home/${USER}/anaconda3/envs/simmim/bin/python")
++ local candidates
++ for p in '"${candidates[@]}"'
++ '[' -x /home/am6949/.conda/envs/simmim/bin/python ']'
++ echo /home/am6949/.conda/envs/simmim/bin/python
++ return 0
+ PY=/home/am6949/.conda/envs/simmim/bin/python
+ echo '[INFO] python candidate: /home/am6949/.conda/envs/simmim/bin/python'


[INFO] python candidate: /home/am6949/.conda/envs/simmim/bin/python


+ test -n /home/am6949/.conda/envs/simmim/bin/python
+ PRE_CFG=/engram/nklab/am6949/SimMIM_ver2/configs/vit_base__800ep/simmim_pretrain__vit_base__img224__800ep.yaml
+ FT_CFG=/engram/nklab/am6949/SimMIM_ver2/configs/vit_base__800ep/simmim_finetune__vit_base__img224__800ep.yaml
+ PRE_MAIN=/engram/nklab/am6949/SimMIM_ver2/main_simmim_blur.py
+ FT_MAIN=/engram/nklab/am6949/SimMIM_ver2/main_finetune_blur.py
+ IMNET_ROOT=/share/data/imagenet-pytorch
+ SWEEP_OUT=/engram/nklab/am6949/SimMIM_ver2/model_weights_ver4/SimMIM_sweeps
+ FT_OUT=/engram/nklab/am6949/SimMIM_ver2/model_weights_ver4/finetune
+ mkdir -p /engram/nklab/am6949/SimMIM_ver2/model_weights_ver4/SimMIM_sweeps /engram/nklab/am6949/SimMIM_ver2/model_weights_ver4/finetune
+ SEED=0
+ DEPTHS=(6 3)
+ RATIOS=(0.65 0.95)
+ BATCH_SIZE=64
+ GAUSS_SCALE=2.0
+ GAUSS_SIGMA_CI_32=1.0
+ PRE_EPOCHS=100
+ FT_EPOCHS=100
+ D6_PRE_WARMUP_EPOCHS=10
+ D6_PRE_BASE_LR=2.0e-4
+ D6_PRE_WARMUP_LR=1.0e-6
+ D6_PRE_DROP_PATH=0.10
+ D3_PRE_WARMUP_EPOCHS=10
+ D

[INFO] JOBFILES:
  /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_d6_gauss_nocut_loop.sh
  /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_d3_gauss_nocut_loop.sh


+ IDS=()
+ for jf in '"${JOBFILES[@]}"'
+ test -f /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_d6_gauss_nocut_loop.sh
+ IDS+=("$(sbatch -p nklab --parsable "${jf}")")
++ sbatch -p nklab --parsable /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_d6_gauss_nocut_loop.sh
+ for jf in '"${JOBFILES[@]}"'
+ test -f /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_d3_gauss_nocut_loop.sh
+ IDS+=("$(sbatch -p nklab --parsable "${jf}")")
++ sbatch -p nklab --parsable /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_d3_gauss_nocut_loop.sh
+ set +x


Submitted 2 jobs on ax17 (seed=0):
  d6_gauss_nocut_loop : 5725864
    tail -n 200 -F /engram/nklab/am6949/SimMIM_ver2/logs/d6_gauss_nocut_loop_5725864.log
    tail -n 200 -F /engram/nklab/am6949/SimMIM_ver2/logs/d6_gauss_nocut_loop_5725864.err
  d3_gauss_nocut_loop : 5725865
    tail -n 200 -F /engram/nklab/am6949/SimMIM_ver2/logs/d3_gauss_nocut_loop_5725865.log
    tail -n 200 -F /engram/nklab/am6949/SimMIM_ver2/logs/d3_gauss_nocut_loop_5725865.err


In [3]:
%%bash
export WANDB_API_KEY="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"
set -euo pipefail
PROJ="/engram/nklab/am6949/SimMIM_ver2"
LOGDIR="${PROJ}/logs"
mkdir -p "${LOGDIR}"

set -x

find_simmim_python () {
  local candidates=(
    "${HOME}/.conda/envs/simmim/bin/python"
    "/home/${USER}/.conda/envs/simmim/bin/python"
    "/home/${USER}/miniconda3/envs/simmim/bin/python"
    "/home/${USER}/anaconda3/envs/simmim/bin/python"
  )
  for p in "${candidates[@]}"; do
    if [ -x "$p" ]; then
      echo "$p"
      return 0
    fi
  done
  return 1
}

PY="$(find_simmim_python || true)"
echo "[INFO] python candidate: ${PY:-<none>}"
test -n "${PY}" || { echo "[FATAL] Could not find simmim python." >&2; exit 127; }

# ----------------------------
# Configs / scripts
# ----------------------------
FT_CFG="${PROJ}/configs/vit_base__800ep/simmim_finetune__vit_base__img224__800ep.yaml"
FT_MAIN="${PROJ}/main_finetune_blur.py"

IMNET_ROOT="/share/data/imagenet-pytorch"

# outputs
SWEEP_OUT="${PROJ}/model_weights_ver4/SimMIM_sweeps"
FT_OUT="${PROJ}/model_weights_ver4/finetune"
mkdir -p "${FT_OUT}"

# ----------------------------
# Sweep definition
# ----------------------------
SEED="0"
DEPTHS=(6 3)
RATIO="0.65"

# per-GPU batch size
BATCH_SIZE="64"

GAUSS_SCALE="2.0"
GAUSS_SIGMA_CI_32="1.0"

FT_EPOCHS="100"

# ----------------------------
# depth-specific FINETUNE hyperparams
# ----------------------------
D6_WARMUP_EPOCHS="20"
D6_BASE_LR="1.50e-3"
D6_WARMUP_LR="3.0e-6"
D6_MIN_LR="1.0e-6"
D6_WEIGHT_DECAY="0.05"
D6_LAYER_DECAY="0.65"
D6_DROP_PATH="0.10"

D3_WARMUP_EPOCHS="${D6_WARMUP_EPOCHS}"
D3_BASE_LR="${D6_BASE_LR}"
D3_WARMUP_LR="${D6_WARMUP_LR}"
D3_MIN_LR="${D6_MIN_LR}"
D3_WEIGHT_DECAY="${D6_WEIGHT_DECAY}"
D3_LAYER_DECAY="${D6_LAYER_DECAY}"
D3_DROP_PATH="${D6_DROP_PATH}"

# ----------------------------
# finetune-only dataloader overrides
# ----------------------------
FT_NUM_WORKERS="8"
FT_PIN_MEMORY="True"

DEPTHS_STR="${DEPTHS[*]}"

write_job () {
  local JOBTAG="gauss_nocut_ft_only_r065_2gpu"
  local JOBFILE="${PROJ}/JOB_DDP_${JOBTAG}.sh"

  cat > "${JOBFILE}" <<'EOS'
#!/bin/bash
#SBATCH --job-name=__JOBTAG__
#SBATCH --partition=nklab
#SBATCH --gres=gpu:2
#SBATCH --cpus-per-task=16
#SBATCH --nodelist=ax26
#SBATCH --mem=64G
#SBATCH --time=10-00:00:00
#SBATCH --output=__LOGDIR__/__JOBTAG___%j.log
#SBATCH --error=__LOGDIR__/__JOBTAG___%j.err

set -euo pipefail
unset MASTER_ADDR MASTER_PORT

echo "[INFO] JOBTAG=__JOBTAG__"
echo "[INFO] DEPTHS=__DEPTHS_STR__"
echo "[INFO] SEED=__SEED__"
echo "[INFO] RATIO=__RATIO__"
echo "[INFO] BATCH_SIZE(per GPU)=__BATCH_SIZE__"
echo "[INFO] CORR=gauss (FFT) NO-CUTOFF"
echo "[INFO] sigma_ci(32)=__GAUSS_SIGMA_CI_32__  scale=__GAUSS_SCALE__"
echo "[INFO] FT_NUM_WORKERS=__FT_NUM_WORKERS__  FT_PIN_MEMORY=__FT_PIN_MEMORY__"
echo "[INFO] Host=$(hostname)"
echo "[INFO] SLURM_JOB_ID=${SLURM_JOB_ID}"
echo "[INFO] Date=$(date)"
echo ""

find_simmim_python () {
  local candidates=(
    "${HOME}/.conda/envs/simmim/bin/python"
    "/home/${USER}/.conda/envs/simmim/bin/python"
    "/home/${USER}/miniconda3/envs/simmim/bin/python"
    "/home/${USER}/anaconda3/envs/simmim/bin/python"
  )
  for p in "${candidates[@]}"; do
    if [ -x "$p" ]; then
      echo "$p"
      return 0
    fi
  done
  return 1
}

PY="$(find_simmim_python || true)"
if [ -z "${PY}" ]; then
  echo "[FATAL] Could not find simmim python." >&2
  exit 127
fi
echo "[INFO] Using PY=${PY}"
"${PY}" -V || true
echo ""

nvidia-smi || true
echo ""

NGPU="${SLURM_GPUS_ON_NODE:-2}"
if [ -n "${SLURM_JOB_GPUS:-}" ]; then
  NGPU="$("${PY}" - <<'PY'
import os
s=os.environ.get("SLURM_JOB_GPUS","").strip()
if not s:
    print(2)
else:
    parts=[p for p in s.replace(" ",",").split(",") if p!=""]
    print(len(parts) if parts else 2)
PY
)"
fi
echo "[INFO] SLURM_GPUS_ON_NODE=${SLURM_GPUS_ON_NODE:-<unset>}  SLURM_JOB_GPUS=${SLURM_JOB_GPUS:-<unset>}  -> NGPU=${NGPU}"

export OMP_NUM_THREADS=1
export MKL_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1
export NUMEXPR_NUM_THREADS=1
export PYTHONPATH="__PROJ__:${PYTHONPATH:-}"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

export PROJ="__PROJ__"
export FT_CFG="__FT_CFG__"
export FT_MAIN="__FT_MAIN__"
export IMNET_ROOT="__IMNET_ROOT__"
export SWEEP_OUT="__SWEEP_OUT__"
export FT_OUT="__FT_OUT__"

export GAUSS_SCALE="__GAUSS_SCALE__"
export GAUSS_SIGMA_CI_32="__GAUSS_SIGMA_CI_32__"

export FT_EPOCHS="__FT_EPOCHS__"
export SEED="__SEED__"
export BATCH_SIZE="__BATCH_SIZE__"
export RATIO="__RATIO__"
export DEPTHS_STR="__DEPTHS_STR__"

export D6_WARMUP_EPOCHS="__D6_WARMUP_EPOCHS__"
export D6_BASE_LR="__D6_BASE_LR__"
export D6_WARMUP_LR="__D6_WARMUP_LR__"
export D6_MIN_LR="__D6_MIN_LR__"
export D6_WEIGHT_DECAY="__D6_WEIGHT_DECAY__"
export D6_LAYER_DECAY="__D6_LAYER_DECAY__"
export D6_DROP_PATH="__D6_DROP_PATH__"

export D3_WARMUP_EPOCHS="__D3_WARMUP_EPOCHS__"
export D3_BASE_LR="__D3_BASE_LR__"
export D3_WARMUP_LR="__D3_WARMUP_LR__"
export D3_MIN_LR="__D3_MIN_LR__"
export D3_WEIGHT_DECAY="__D3_WEIGHT_DECAY__"
export D3_LAYER_DECAY="__D3_LAYER_DECAY__"
export D3_DROP_PATH="__D3_DROP_PATH__"

export FT_NUM_WORKERS="__FT_NUM_WORKERS__"
export FT_PIN_MEMORY="__FT_PIN_MEMORY__"

mkdir -p "${FT_OUT}"
cd "${PROJ}"

SIGMA_CI="$("${PY}" - <<'PY'
import os
s32 = float(os.environ["GAUSS_SIGMA_CI_32"])
scale = float(os.environ["GAUSS_SCALE"])
print(s32 * scale)
PY
)"
echo "[INFO] FFT-Gauss scaled: sigma_ci=${SIGMA_CI}  cutoff_ci=None (not overridden)"

for DEPTH in __DEPTHS_STR__; do
  case "${DEPTH}" in
    6)
      FT_WARMUP_EPOCHS="${D6_WARMUP_EPOCHS}"
      FT_BASE_LR="${D6_BASE_LR}"
      FT_WARMUP_LR="${D6_WARMUP_LR}"
      FT_MIN_LR="${D6_MIN_LR}"
      FT_WEIGHT_DECAY="${D6_WEIGHT_DECAY}"
      FT_LAYER_DECAY="${D6_LAYER_DECAY}"
      FT_DROP_PATH="${D6_DROP_PATH}"
      ;;
    3)
      FT_WARMUP_EPOCHS="${D3_WARMUP_EPOCHS}"
      FT_BASE_LR="${D3_BASE_LR}"
      FT_WARMUP_LR="${D3_WARMUP_LR}"
      FT_MIN_LR="${D3_MIN_LR}"
      FT_WEIGHT_DECAY="${D3_WEIGHT_DECAY}"
      FT_LAYER_DECAY="${D3_LAYER_DECAY}"
      FT_DROP_PATH="${D3_DROP_PATH}"
      ;;
    *)
      echo "[FATAL] Unsupported DEPTH=${DEPTH} (expected 3 or 6)" >&2
      exit 2
      ;;
  esac

  RTAG="$(echo "${RATIO}" | sed 's/0\.//; s/\.//g')"
  FT_WANDB_NAME="blur_nocut_d${DEPTH}_r${RTAG}_ft"

  RUNNAME="vitb_img224_seed${SEED}_d${DEPTH}_r${RATIO}_gauss_nocut_s${SIGMA_CI}"
  PRE_OUT_DIR="${SWEEP_OUT}/${RUNNAME}"
  FT_OUT_DIR="${FT_OUT}/${RUNNAME}__FT_ep${FT_EPOCHS}"

  CKPT=""
  if [ -f "${PRE_OUT_DIR}/simmim_pretrain/sweep/best.pt" ]; then
    CKPT="${PRE_OUT_DIR}/simmim_pretrain/sweep/best.pt"
  elif [ -f "${PRE_OUT_DIR}/simmim_pretrain/sweep/best.pth" ]; then
    CKPT="${PRE_OUT_DIR}/simmim_pretrain/sweep/best.pth"
  else
    CKPT="$(ls -1 "${PRE_OUT_DIR}"/simmim_pretrain/sweep/ckpt_epoch_* 2>/dev/null | tail -n 1 || true)"
  fi
  test -f "${CKPT}" || { echo "[FATAL] Missing pretrained checkpoint: ${CKPT}" >&2; exit 3; }

  mkdir -p "${FT_OUT_DIR}"

  echo "============================================================"
  echo "[RUN] DEPTH=${DEPTH}  MASK_RATIO=${RATIO}  CORR=gauss (FFT)  NO-CUTOFF  FINETUNE ONLY  SEED=${SEED}"
  echo "[CKPT] ${CKPT}"
  echo "[W&B] FT=${FT_WANDB_NAME}"
  echo "[RESOLVED] FT: warmup_epochs=${FT_WARMUP_EPOCHS} base_lr=${FT_BASE_LR} warmup_lr=${FT_WARMUP_LR} min_lr=${FT_MIN_LR} wd=${FT_WEIGHT_DECAY} layer_decay=${FT_LAYER_DECAY} drop_path=${FT_DROP_PATH}"
  echo "[RESOLVED] FT loader: num_workers=${FT_NUM_WORKERS} pin_memory=${FT_PIN_MEMORY}"
  echo "[RESOLVED] DDP GPUs=${NGPU} batch_size_per_gpu=${BATCH_SIZE}"
  echo "============================================================"

  OPTS_FT=(
    SEED "${SEED}"
    TRAIN.EPOCHS "${FT_EPOCHS}"

    MODEL.TYPE vit
    MODEL.VIT.DEPTH "${DEPTH}"

    TRAIN.WARMUP_EPOCHS "${FT_WARMUP_EPOCHS}"
    TRAIN.BASE_LR "${FT_BASE_LR}"
    TRAIN.WARMUP_LR "${FT_WARMUP_LR}"
    TRAIN.MIN_LR "${FT_MIN_LR}"
    TRAIN.WEIGHT_DECAY "${FT_WEIGHT_DECAY}"
    TRAIN.LAYER_DECAY "${FT_LAYER_DECAY}"
    MODEL.DROP_PATH_RATE "${FT_DROP_PATH}"

    DATA.NUM_WORKERS "${FT_NUM_WORKERS}"
    DATA.PIN_MEMORY "${FT_PIN_MEMORY}"
  )

  echo "[INFO] FT --opts: ${OPTS_FT[*]}"

  "${PY}" -m torch.distributed.run --standalone --nproc_per_node="${NGPU}" "${FT_MAIN}" \
    --cfg "${FT_CFG}" \
    --data-path "${IMNET_ROOT}" \
    --pretrained "${CKPT}" \
    --output "${FT_OUT_DIR}" \
    --tag finetune \
    --batch-size "${BATCH_SIZE}" \
    --wandb-run-name "${FT_WANDB_NAME}" \
    --amp-opt-level O1 \
    --opts ${OPTS_FT[@]}

  echo "[DONE] DEPTH=${DEPTH} finished."
done

echo "[DONE] __JOBTAG__ finished all depths."
EOS

  sed -i \
    -e "s|__PROJ__|${PROJ}|g" \
    -e "s|__LOGDIR__|${LOGDIR}|g" \
    -e "s|__FT_CFG__|${FT_CFG}|g" \
    -e "s|__FT_MAIN__|${FT_MAIN}|g" \
    -e "s|__IMNET_ROOT__|${IMNET_ROOT}|g" \
    -e "s|__SWEEP_OUT__|${SWEEP_OUT}|g" \
    -e "s|__FT_OUT__|${FT_OUT}|g" \
    -e "s|__GAUSS_SCALE__|${GAUSS_SCALE}|g" \
    -e "s|__GAUSS_SIGMA_CI_32__|${GAUSS_SIGMA_CI_32}|g" \
    -e "s|__FT_EPOCHS__|${FT_EPOCHS}|g" \
    -e "s|__SEED__|${SEED}|g" \
    -e "s|__BATCH_SIZE__|${BATCH_SIZE}|g" \
    -e "s|__RATIO__|${RATIO}|g" \
    -e "s|__DEPTHS_STR__|${DEPTHS_STR}|g" \
    -e "s|__D6_WARMUP_EPOCHS__|${D6_WARMUP_EPOCHS}|g" \
    -e "s|__D6_BASE_LR__|${D6_BASE_LR}|g" \
    -e "s|__D6_WARMUP_LR__|${D6_WARMUP_LR}|g" \
    -e "s|__D6_MIN_LR__|${D6_MIN_LR}|g" \
    -e "s|__D6_WEIGHT_DECAY__|${D6_WEIGHT_DECAY}|g" \
    -e "s|__D6_LAYER_DECAY__|${D6_LAYER_DECAY}|g" \
    -e "s|__D6_DROP_PATH__|${D6_DROP_PATH}|g" \
    -e "s|__D3_WARMUP_EPOCHS__|${D3_WARMUP_EPOCHS}|g" \
    -e "s|__D3_BASE_LR__|${D3_BASE_LR}|g" \
    -e "s|__D3_WARMUP_LR__|${D3_WARMUP_LR}|g" \
    -e "s|__D3_MIN_LR__|${D3_MIN_LR}|g" \
    -e "s|__D3_WEIGHT_DECAY__|${D3_WEIGHT_DECAY}|g" \
    -e "s|__D3_LAYER_DECAY__|${D3_LAYER_DECAY}|g" \
    -e "s|__D3_DROP_PATH__|${D3_DROP_PATH}|g" \
    -e "s|__FT_NUM_WORKERS__|${FT_NUM_WORKERS}|g" \
    -e "s|__FT_PIN_MEMORY__|${FT_PIN_MEMORY}|g" \
    -e "s|__JOBTAG__|${JOBTAG}|g" \
    "${JOBFILE}"

  chmod +x "${JOBFILE}"
  echo "${JOBFILE}"
}

JOBFILE="$(write_job)"

echo "[INFO] JOBFILE:"
printf '  %s\n' "${JOBFILE}"

test -f "${JOBFILE}" || { echo "[FATAL] Missing jobfile: ${JOBFILE}" >&2; exit 4; }
JOBID="$(sbatch -p nklab --parsable "${JOBFILE}")"

set +x
echo "Submitted 1 finetune-only 2-GPU job on ax26 (seed=${SEED}):"
echo "  depths=${DEPTHS_STR}"
echo "  ratio=${RATIO}"
echo "  DATA.NUM_WORKERS=${FT_NUM_WORKERS}"
echo "  DATA.PIN_MEMORY=${FT_PIN_MEMORY}"
echo "  ${JOBID}"
echo "    tail -n 200 -F ${LOGDIR}/gauss_nocut_ft_only_r065_2gpu_${JOBID}.log"
echo "    tail -n 200 -F ${LOGDIR}/gauss_nocut_ft_only_r065_2gpu_${JOBID}.err"

++ find_simmim_python
++ candidates=("${HOME}/.conda/envs/simmim/bin/python" "/home/${USER}/.conda/envs/simmim/bin/python" "/home/${USER}/miniconda3/envs/simmim/bin/python" "/home/${USER}/anaconda3/envs/simmim/bin/python")
++ local candidates
++ for p in '"${candidates[@]}"'
++ '[' -x /home/am6949/.conda/envs/simmim/bin/python ']'
++ echo /home/am6949/.conda/envs/simmim/bin/python
++ return 0
+ PY=/home/am6949/.conda/envs/simmim/bin/python
+ echo '[INFO] python candidate: /home/am6949/.conda/envs/simmim/bin/python'
+ test -n /home/am6949/.conda/envs/simmim/bin/python


[INFO] python candidate: /home/am6949/.conda/envs/simmim/bin/python


+ FT_CFG=/engram/nklab/am6949/SimMIM_ver2/configs/vit_base__800ep/simmim_finetune__vit_base__img224__800ep.yaml
+ FT_MAIN=/engram/nklab/am6949/SimMIM_ver2/main_finetune_blur.py
+ IMNET_ROOT=/share/data/imagenet-pytorch
+ SWEEP_OUT=/engram/nklab/am6949/SimMIM_ver2/model_weights_ver4/SimMIM_sweeps
+ FT_OUT=/engram/nklab/am6949/SimMIM_ver2/model_weights_ver4/finetune
+ mkdir -p /engram/nklab/am6949/SimMIM_ver2/model_weights_ver4/finetune
+ SEED=0
+ DEPTHS=(6 3)
+ RATIO=0.65
+ BATCH_SIZE=64
+ GAUSS_SCALE=2.0
+ GAUSS_SIGMA_CI_32=1.0
+ FT_EPOCHS=100
+ D6_WARMUP_EPOCHS=20
+ D6_BASE_LR=1.50e-3
+ D6_WARMUP_LR=3.0e-6
+ D6_MIN_LR=1.0e-6
+ D6_WEIGHT_DECAY=0.05
+ D6_LAYER_DECAY=0.65
+ D6_DROP_PATH=0.10
+ D3_WARMUP_EPOCHS=20
+ D3_BASE_LR=1.50e-3
+ D3_WARMUP_LR=3.0e-6
+ D3_MIN_LR=1.0e-6
+ D3_WEIGHT_DECAY=0.05
+ D3_LAYER_DECAY=0.65
+ D3_DROP_PATH=0.10
+ FT_NUM_WORKERS=8
+ FT_PIN_MEMORY=True
+ DEPTHS_STR='6 3'
++ write_job
++ local JOBTAG=gauss_nocut_ft_only_r065_2gpu
++ local JOBFILE=/engram/nklab/am6

[INFO] JOBFILE:
  /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_gauss_nocut_ft_only_r065_2gpu.sh


+ test -f /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_gauss_nocut_ft_only_r065_2gpu.sh
++ sbatch -p nklab --parsable /engram/nklab/am6949/SimMIM_ver2/JOB_DDP_gauss_nocut_ft_only_r065_2gpu.sh
+ JOBID=5727682
+ set +x


Submitted 1 finetune-only 2-GPU job on ax26 (seed=0):
  depths=6 3
  ratio=0.65
  DATA.NUM_WORKERS=8
  DATA.PIN_MEMORY=True
  5727682
    tail -n 200 -F /engram/nklab/am6949/SimMIM_ver2/logs/gauss_nocut_ft_only_r065_2gpu_5727682.log
    tail -n 200 -F /engram/nklab/am6949/SimMIM_ver2/logs/gauss_nocut_ft_only_r065_2gpu_5727682.err
